**Code Vision OCR Submodel**

**Importing Dependencies**

In [ ]:
!pip install Levenshtein
import Levenshtein

In [ ]:
!pip install --upgrade google-cloud-vision
from google.cloud import vision

In [ ]:
import io
import os
import pandas as pd
from PIL import Image, ImageEnhance
import re
import cv2
import numpy as np

In [ ]:
!unzip /content/screenshots-gen-ai.zip

Archive:  /content/screenshots-gen-ai.zip
   creating: screenshots-gen-ai/
  inflating: screenshots-gen-ai/.DS_Store  
   creating: screenshots-gen-ai/python/
   creating: screenshots-gen-ai/java/
  inflating: screenshots-gen-ai/python/88.png  
  inflating: screenshots-gen-ai/python/77.png  
  inflating: screenshots-gen-ai/python/63.png  
  inflating: screenshots-gen-ai/python/62.png  
  inflating: screenshots-gen-ai/python/76.png  
  inflating: screenshots-gen-ai/python/89.png  
  inflating: screenshots-gen-ai/python/60.png  
  inflating: screenshots-gen-ai/python/74.png  
  inflating: screenshots-gen-ai/python/48.png  
  inflating: screenshots-gen-ai/python/49.png  
  inflating: screenshots-gen-ai/python/75.png  
  inflating: screenshots-gen-ai/python/61.png  
  inflating: screenshots-gen-ai/python/59.png  
  inflating: screenshots-gen-ai/python/65.png  
  inflating: screenshots-gen-ai/python/71.png  
  inflating: screenshots-gen-ai/python/.DS_Store  
  inflating: screenshots-gen-ai/

In [ ]:
!unzip /content/gen-ai-test.zip
!unzip /content/gen-ai-test-cs.zip

Archive:  /content/gen-ai-test.zip
   creating: gen-ai-test/
  inflating: gen-ai-test/.DS_Store   
  inflating: __MACOSX/gen-ai-test/._.DS_Store  
  inflating: gen-ai-test/8.png       
  inflating: __MACOSX/gen-ai-test/._8.png  
  inflating: gen-ai-test/9.png       
  inflating: __MACOSX/gen-ai-test/._9.png  
  inflating: gen-ai-test/14.png      
  inflating: __MACOSX/gen-ai-test/._14.png  
  inflating: gen-ai-test/15.png      
  inflating: __MACOSX/gen-ai-test/._15.png  
  inflating: gen-ai-test/17.png      
  inflating: __MACOSX/gen-ai-test/._17.png  
  inflating: gen-ai-test/16.png      
  inflating: __MACOSX/gen-ai-test/._16.png  
  inflating: gen-ai-test/12.png      
  inflating: __MACOSX/gen-ai-test/._12.png  
  inflating: gen-ai-test/13.png      
  inflating: __MACOSX/gen-ai-test/._13.png  
  inflating: gen-ai-test/11.png      
  inflating: __MACOSX/gen-ai-test/._11.png  
  inflating: gen-ai-test/10.png      
  inflating: __MACOSX/gen-ai-test/._10.png  
  inflating: gen-ai-test/

In [ ]:
def extract_code_from_image(image_path):
  os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = '/content/ocr-for-gen-ai-fac0fbc83a73.json'
  client = vision.ImageAnnotatorClient()

  with io.open(image_path,'rb') as image_file:
    content = image_file.read()

  image = vision.Image(content=content)
  response = client.document_text_detection(image=image)

  text = response.full_text_annotation.text
  return text

**Preprocessing Functions**

In [ ]:
def binarize_image(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    binary = cv2.adaptiveThreshold(img, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                   cv2.THRESH_BINARY, 11, 2)
    #cv2.imwrite("binarized.png", binary)
    return "binarized.png"

def preprocess_image(input_path, output_path="processed.png"):
    img = Image.open(input_path).convert("L")  # grayscale
    enhancer = ImageEnhance.Contrast(img)
    enhanced_image = enhancer.enhance(2.0)
    enhanced_image.save(output_path)
    return output_path

**NLD**

In [ ]:
def normalized_levenshtein(s1, s2):

    dist = Levenshtein.distance(s1, s2)
    max_len = max(len(s1), len(s2))

    return dist / max_len if max_len > 0 else 0.0

def log_nld(index, nld, log_file="log.txt"):
    with open(log_file, "a") as f:
        f.write(f"Image {index} | NLD: {nld:.4f}\n")

def normalize(code):
    return ''.join(code.strip().split())

In [ ]:
def run(language,clean):
    df = pd.read_csv('ocr_data.csv') # ground truth code from csv dataset

    image_folder = f"/content/screenshots-gen-ai/{language}"

    nld_iter = []
    exact,accurate = 0,0

    for i in range(100):
        image_path = os.path.join(image_folder,f'{i}.png')
        #print(f"Using image {i}: {image_path}")

        if clean:
            cleaned_image = preprocess_image(image_path) # preprocessing
            #cleaned_image = binarize_image(image_path)
            code_text = extract_code_from_image(cleaned_image) #code extraction
        else:
            code_text = extract_code_from_image(image_path)
        cleaned_code = re.sub(r"\s+", "", code_text) #no whitespace

        #groundTruth code from csv
        if language == 'python':
            code_gt = df.iloc[i]['Code'] # for python
        elif language == 'java':
            code_gt = df.iloc[100+i]['Code'] # for java

        code_gt = code_gt.replace(r"\n", "\n")
        code_groundTruth = re.sub(r"\s+", "", code_gt) #no whitespace

        nld = normalized_levenshtein(normalize(cleaned_code),normalize(code_groundTruth))
        if nld >= 0.5:
            print(f"Image {i}.png → NLD: {nld:.4f}")
            print('code:',cleaned_code)
            print('groundTruth:',code_groundTruth)
        elif nld <= 0.05:
            exact += 1
            accurate += 1
        elif nld <= 0.1:
            accurate += 1
        nld_iter.append(nld)
        log_nld(i,nld=nld)

    avg = sum(nld_iter)/len(nld_iter)
    return nld_iter, avg, exact, accurate

In [ ]:
nld_iter, avg, exact, accurate = run('java',clean=True)
print(nld_iter)
print('Average NLD:', sum(nld_iter)/len(nld_iter))
print('Exact:', exact)
print('Accurate:', accurate)

[0.011627906976744186, 0.052941176470588235, 0.002145922746781116, 0.06167400881057269, 0.08670520231213873, 0.046232876712328765, 0.022091310751104567, 0.026455026455026454, 0.01411764705882353, 0.035830618892508145, 0.1308016877637131, 0.015625, 0.021108179419525065, 0.02557544757033248, 0.005952380952380952, 0.019490254872563718, 0.031067961165048542, 0.0390104662226451, 0.014644351464435146, 0.0035087719298245615, 0.06418918918918919, 0.005502063273727648, 0.020348837209302327, 0.006711409395973154, 0.02185792349726776, 0.0, 0.0, 0.017994858611825194, 0.007042253521126761, 0.006051437216338881, 0.010810810810810811, 0.010416666666666666, 0.05333333333333334, 0.09502762430939227, 0.018726591760299626, 0.06374501992031872, 0.019444444444444445, 0.056962025316455694, 0.016483516483516484, 0.0084985835694051, 0.027989821882951654, 0.012403100775193798, 0.007692307692307693, 0.014245014245014245, 0.00992063492063492, 0.02348993288590604, 0.041666666666666664, 0.046511627906976744, 0.052

In [ ]:
def run_specific(language,clean):
    df = pd.read_csv('ocr_data.csv') # ground truth code from csv dataset

    image_folder = f"/content/screenshots-gen-ai/{language}"

    nld_iter = []
    exact,accurate = 0,0

    i = np.random.randint(0,99)

    image_path = os.path.join(image_folder,f'{i}.png')
    #print(f"Using image {i}: {image_path}")

    if clean:
        cleaned_image = preprocess_image(image_path) # preprocessing
            #cleaned_image = binarize_image(image_path)
        code_text = extract_code_from_image(cleaned_image) #code extraction
    else:
        code_text = extract_code_from_image(image_path)
    cleaned_code = re.sub(r"\s+", "", code_text) #no whitespace

    return normalize(cleaned_code)

In [ ]:
print(run_specific('java',clean=True))

//tryeachpossiblestartindexfromhaystackpublicintstrStr(Stringhaystack,Stringneedle){for(intindex=0;index<=haystack.length()-needle.length();index++){inti=index;intj=0;while(j<needle.length()&&haystack.charAt(i)==needle.charAt(j)){i++;j++;}if(j==needle.length()){//foundreturnindex;}}return-1;}


In [ ]:
def runJavaTest(clean):

    image_folder = f"/content/gen-ai-test"

    javaCodeTestList = []
    exact,accurate = 0,0

    for i in range(20):
        image_path = os.path.join(image_folder,f'{i}.png')
        #print(f"Using image {i}: {image_path}")

        if clean:
            cleaned_image = preprocess_image(image_path) # preprocessing
            #cleaned_image = binarize_image(image_path)
            code_text = extract_code_from_image(cleaned_image) #code extraction
        else:
            code_text = extract_code_from_image(image_path)
        cleaned_code = re.sub(r"\s+", "", code_text) #no whitespace
        javaCodeTestList.append(normalize(cleaned_code))

    return javaCodeTestList

In [ ]:
javaCodeTestList = runJavaTest(clean=True)
print(javaCodeTestList)

['publicvoidserialize(LittleEndianOutputout){out.writeShort(field_1_vcenter);}', 'publicvoidaddAll(BlockList<T>src){}if(src.size==0)return;intsrcDirIdx=0;for(;srcDirIdx<src.tailDirIdx;srcDirIdx++)addAll(src.directory[srcDirIdx],0,BLOCK_SIZE);if(src.tailBlkIdx!=0)addAll(src.tailBlock,0,src.tailBlkIdx);', 'publicvoidwriteByte(byteb){if(upto==blockSize){}}if(currentBlock!=null){}addBlock(currentBlock);currentBlock=newbyte[blockSize];upto=0;currentBlock[upto++]=b;', 'publicObjectIdgetObjectId(){returnobjectId;}', 'publicDeleteDomainEntryResultdeleteDomainEntry(DeleteDomainEntryRequestrequest){request=beforeClientExecution(request);returnexecuteDeleteDomainEntry(request);}', 'publiclongramBytesUsed(){}return((termOffsets!=null)?termOffsets.ramBytesUsed():0)+((termsDictOffsets!=null)?termsDictOffsets.ramBytesUsed():0);', 'publicfinalStringgetFullMessage(){byte[]raw=buffer;intmsgB=RawParseUtils.tagMessage(raw,if(msgB<0){return"";0);returnRawParseUtils.decode(guessEncoding(),raw,msgB,raw.lengt

In [ ]:
def runCSTest(clean):

    image_folder = f"/content/gen-ai-test-cs"

    csCodeTestList = []
    exact,accurate = 0,0

    for i in range(20):
        image_path = os.path.join(image_folder,f'{i}.png')
        #print(f"Using image {i}: {image_path}")

        if clean:
            cleaned_image = preprocess_image(image_path) # preprocessing
            #cleaned_image = binarize_image(image_path)
            code_text = extract_code_from_image(cleaned_image) #code extraction
        else:
            code_text = extract_code_from_image(image_path)
        cleaned_code = re.sub(r"\s+", "", code_text) #no whitespace
        csCodeTestList.append(normalize(cleaned_code))

    return csCodeTestList

In [ ]:
csCodeTestList = runCSTest(clean=True)
print(csCodeTestList)

['publicoverridevoidSerialize(ILittleEndianOutputout1){out1.WriteShort(field_1_vcenter);}', 'publicvirtualvoidAddAll(NGit.Util.BlockList<T>src){if(src.size==0){return;}intsrcDirIdx=0;for(;srcDirIdx<src.tailDirIdx%;srcDirIdx++){}AddAll(src.directory[srcDirIdx],0,BLOCK_SIZE);if(src.tailBlkIdx!=0){}}AddAll(src.tailBlock,0,src.tailBlkIdx);', '{publicoverridevoidWriteByte(byteb)if(outerInstance.upto==={outerInstance.blockSize)if(outerInstance.currentBlock!=null){outerInstance.blocks.Add(outerInstance.currentBlock);outerInstance.blockEnd.Add(outerInstance.upto);}}outerInstance.currentBlock=newbyte[outerInstance.blockSize];outerInstance.upto=0;}outerInstance.currentBlock[outerInstance.upto++]=b;', 'publicvirtualObjectIdGetObjectId(){returnobjectId;}', 'publicvirtualDeleteDomainEntryResponseDeleteDomainEntry(DeleteDomainEntryRequestrequest){varoptions=newInvokeOptions{};RequestMarshaller=DeleteDomainEntryRequestMarshaller.Instance,ResponseUnmarshaller=DeleteDomainEntryResponseUnmarshaller.Inst

**Code Translation Submodel**

**Installing Library Dependencies**

In [ ]:
!pip install datasets
!pip install evaluate

**Importing Dataset**

In [ ]:
from datasets import load_dataset

dataset = load_dataset("google/code_x_glue_cc_code_to_code_trans")
print(dataset.column_names)
print(dataset['train'][0])

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/170k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10300 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'train': ['id', 'java', 'cs'], 'validation': ['id', 'java', 'cs'], 'test': ['id', 'java', 'cs']}
{'id': 0, 'java': 'public ListSpeechSynthesisTasksResult listSpeechSynthesisTasks(ListSpeechSynthesisTasksRequest request) {request = beforeClientExecution(request);return executeListSpeechSynthesisTasks(request);}\n', 'cs': 'public virtual ListSpeechSynthesisTasksResponse ListSpeechSynthesisTasks(ListSpeechSynthesisTasksRequest request){var options = new InvokeOptions();options.RequestMarshaller = ListSpeechSynthesisTasksRequestMarshaller.Instance;options.ResponseUnmarshaller = ListSpeechSynthesisTasksResponseUnmarshaller.Instance;return Invoke<ListSpeechSynthesisTasksResponse>(request, options);}\n'}


**Baseline Testing & Evaluation** \\
We will be testing two baseline models: CodeT5-Small and CodeT5-Base. For each model, we will be evaluating the model's performance on Java to C# translations and also C# to Java. The functions below define the model loading and evaluation process using the *code_x_glue* test dataset.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

def model_loader(model):
  if model=='base':
    model_name = "Salesforce/codet5-base"
  elif model=='small':
    model_name = "Salesforce/codet5-small"
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)
  return tokenizer, model, device

In [ ]:
import evaluate
def javaToCSharp(tokenizer, model, device):
  predictions = []
  references = []

  print("Generating Java to C# translations ...")
  for example in dataset['test']:
      source_code = example["java"]
      reference_code = example["cs"]

      prompt = f"Translate Java to C#: {source_code}"

      inputs = tokenizer(prompt, return_tensors="pt", padding="max_length",
                        truncation=True, max_length=512)
      inputs = {k: v.to(device) for k, v in inputs.items()}

      outputs = model.generate(**inputs, max_length=512)
      translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

      predictions.append(translated_code)
      references.append(reference_code)

  bleu_metric = evaluate.load("bleu")

  results = bleu_metric.compute(predictions=predictions, references=references)
  print("BLEU Score:", results["bleu"])

  exact_matches = 0
  for pred, ref in zip(predictions, references):
      if pred.strip() == ref.strip():
          exact_matches += 1

  exact_match_score = exact_matches / len(predictions)
  print("Exact Match Score:", exact_match_score)

def cSharpToJava(tokenizer, model, device):
  predictions = []
  references = []

  print("Generating C# to Java translations ...")
  for example in dataset['test']:
      source_code = example["cs"]
      reference_code = example["java"]

      prompt = f"Translate C# to Java: {source_code}"

      inputs = tokenizer(prompt, return_tensors="pt", padding="max_length",
                        truncation=True, max_length=512)
      inputs = {k: v.to(device) for k, v in inputs.items()}

      outputs = model.generate(**inputs, max_length=512)
      translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

      predictions.append(translated_code)
      references.append(reference_code)

  bleu_metric = evaluate.load("bleu")

  results = bleu_metric.compute(predictions=predictions, references=references)
  print("BLEU Score:", results["bleu"])

  exact_matches = 0
  for pred, ref in zip(predictions, references):
      if pred.strip() == ref.strip():
          exact_matches += 1

  exact_match_score = exact_matches / len(predictions)
  print("Exact Match Score:", exact_match_score)

(i) CodeT5-Base Java → C#

In [ ]:
tokenizer, model, device = model_loader('base')
javaToCSharp(tokenizer, model, device)
cSharpToJava(tokenizer, model, device)

Generating Java to C# translations ...


BLEU Score: 0.01606113600244783
Exact Match Score: 0.0
Generating C# to Java translations ...
BLEU Score: 0.03833964312526373
Exact Match Score: 0.0


(iii) CodeT5-Small Java → C# + C# → Java

In [ ]:
# 4 min 38s run time on T4
tokenizer, model, device = model_loader('small')
javaToCSharp(tokenizer, model, device)
cSharpToJava(tokenizer, model, device)

pytorch_model.bin:  95%|#########5| 231M/242M [00:00<?, ?B/s]

Generating Java to C# translations ...


model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

BLEU Score: 0.0006631554077165196
Exact Match Score: 0.0
Generating C# to Java translations ...
BLEU Score: 0.0003011713015150887
Exact Match Score: 0.0


**Fine Tuning Methods** \\
As seen above, both CodeT5 models perform poorly without any fine-tuning as the original model was built around several tasks across multiple language (ie. code generation, translation, etc.). In order to be effective for our objective, we explore the following fine-tuning methods: Top-K Layer Fine-Tune and Low Rank Adaptation (LoRA).

Top-K Layer Fine-Tune

In [ ]:
model_name = "Salesforce/codet5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# layer freeze
for param in model.parameters():
    param.requires_grad = False

K = 8  # adjust for number of layers

# layer unfreeze
for layer in model.encoder.block[-K:]:
    for param in layer.parameters():
        param.requires_grad = True
for layer in model.decoder.block[-K:]:
    for param in layer.parameters():
        param.requires_grad = True
for param in model.shared.parameters():
    param.requires_grad = True
for param in model.lm_head.parameters():
    param.requires_grad = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

T5ForConditionalGeneration(
  (shared): Embedding(32100, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32100, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=768, out_features=3072, bias=False)
              (wo): Linear(in_features=3072, out_features=768, bias=False)
              (dropout): Dro

In [ ]:
def preprocess(example):
    # for Java to C#
    #inputs = ["translate java to CSharp: " + src for src in example["java"]]
    #targets = example["cs"]

    # for C# to Java
    inputs = ["translate CSharp to java: " + src for src in example["cs"]]
    targets = example["java"]

    model_inputs = tokenizer(
        inputs,
        truncation=True,
        padding="max_length",
        max_length=256
    )

    labels = tokenizer(
        targets,
        truncation=True,
        padding="max_length",
        max_length=256
    ).input_ids

    model_inputs["labels"] = labels
    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(preprocess, batched=True)

Map:   0%|          | 0/10300 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
### Top K training
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir=f"./codet5-topk-ft{K}",
    run_name=f"./codet5-topk-ft",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=5e-4, #original 5e-5
    num_train_epochs=3,
    logging_dir="./logs",
    fp16=True,
    save_steps=500,
    logging_steps=100,
    eval_steps=500,
    save_total_limit=2
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

trainer.train()
trainer.evaluate()

<ipython-input-19-81267605b1af>:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lhsu2 (lhsu2-carnegie-mellon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
100,0.207100
200,0.112200
300,0.109100
400,0.096900
500,0.080900
600,0.070000
700,0.071300
800,0.078600
900,0.068200
1000,0.079000


{'eval_loss': 0.04027503728866577,
 'eval_runtime': 8.1414,
 'eval_samples_per_second': 61.414,
 'eval_steps_per_second': 7.738,
 'epoch': 3.0}

In [ ]:
def translate_example(example):
    # Java to CSharp
    #source_code = example['java']
    #input_text = "translate java to CSharp: " + source_code

    # CSharp to Java
    source_code = example['cs']
    input_text = "translate CSharp to java: " + source_code

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=512)
    translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

    example['predicted'] = translated_code
    return example

In [ ]:
def translate_specific_code(codeString):
    # Java to CSharp
    #input_text = "translate java to CSharp: " + codeString

    # CSharp to Java
    input_text = "translate CSharp to java: " + codeString

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=512)
    translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return translated_code

In [ ]:
#java code from ocr
java_code = run_specific('java',clean=True)
print(java_code)
translation = translate_specific_code(java_code)
print(translation)

publicintjump(int()nums){intn=nums.length;int[]dp=newint[n];for(inti=1;i<n;i++){intsteps=Integer.MAX_VALUE;for(intj=i−1;j>=0;j--){if(dp[j]!=Integer.MAX_VALUE&&nums[j]+j>=i){steps=Math.min(steps,dp[j]+1);}=dp[i]steps;}}}returndp[n-1];
public int Jump(int[] dp){intn = ((int)nums.Length);int[] dp = newint[n];for (inti = 1; i < n; i++){intsteps = int.MaxValue;for (intj = i− 1; j >= 0; j--){if (dp[j] != int.MaxValue&&nums[j + j >= i){steps = Math.Min(steps,dp[j + 1]);}=dp[i]steps;}}}return dp[n - 1];}



**Java → C# Full Pipeline Test**

In [ ]:
def translationJavaTest(codeList):
  translationList = []
  for i in range(len(codeList)):
      sourceCodeString = codeList[i]
      translation = translate_specific_code(sourceCodeString)
      translationList.append(translation)
  return translationList

In [ ]:
javaCodeTestList = runJavaTest(clean=True)
refs = translationJavaTest(javaCodeTestList)
preds = pd.read_csv('javaFinalTest.csv')
preds = preds.iloc[:,0].tolist()
print(preds,refs)
exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

['public override void Serialize(ILittleEndianOutput out1){out1.WriteShort(field_1_vcenter);}', 'public virtual void AddAll(NGit.Util.BlockList<T> src){if (src.size == 0){return;}int srcDirIdx = 0;for (; srcDirIdx < src.tailDirIdx; srcDirIdx++){AddAll(src.directory[srcDirIdx], 0, BLOCK_SIZE);}if (src.tailBlkIdx != 0){AddAll(src.tailBlock, 0, src.tailBlkIdx);}}', 'public override void WriteByte(byte b){if (outerInstance.upto == outerInstance.blockSize){if (outerInstance.currentBlock != null){outerInstance.blocks.Add(outerInstance.currentBlock);outerInstance.blockEnd.Add(outerInstance.upto);}outerInstance.currentBlock = new byte[outerInstance.blockSize];outerInstance.upto = 0;}outerInstance.currentBlock[outerInstance.upto++] = (byte)b;}', 'public virtual ObjectId GetObjectId(){return objectId;}', 'public virtual DeleteDomainEntryResponse DeleteDomainEntry(DeleteDomainEntryRequest request){var options = new InvokeOptions();options.RequestMarshaller = DeleteDomainEntryRequestMarshaller.Ins

**C# → Java Full Pipeline Test**

In [ ]:
def translationCSTest(codeList):
  translationList = []
  for i in range(len(codeList)):
      sourceCodeString = codeList[i]
      translation = translate_specific_code(sourceCodeString)
      translationList.append(translation)
  return translationList

In [ ]:
csCodeTestList = runCSTest(clean=True)
refs = translationCSTest(csCodeTestList)
preds = pd.read_csv('csFinalTest.csv')
preds = preds.iloc[:,0].tolist()
print(preds,refs)
exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

['public void serialize(LittleEndianOutput out) {out.writeShort(field_1_vcenter);}', 'public void addAll(BlockList<T> src) {if (src.size == 0)return;int srcDirIdx = 0;for (; srcDirIdx < src.tailDirIdx; srcDirIdx++)addAll(src.directory[srcDirIdx], 0, BLOCK_SIZE);if (src.tailBlkIdx != 0)addAll(src.tailBlock, 0, src.tailBlkIdx);}', 'public void writeByte(byte b) {if (upto == blockSize) {if (currentBlock != null) {addBlock(currentBlock);}currentBlock = new byte[blockSize];upto = 0;}currentBlock[upto++] = b;}', 'public ObjectId getObjectId() {return objectId;}', 'public DeleteDomainEntryResult deleteDomainEntry(DeleteDomainEntryRequest request) {request = beforeClientExecution(request);return executeDeleteDomainEntry(request);}', 'public long ramBytesUsed() {return ((termOffsets!=null)? termOffsets.ramBytesUsed() : 0) +((termsDictOffsets!=null)? termsDictOffsets.ramBytesUsed() : 0);}', 'public final String getFullMessage() {byte[] raw = buffer;int msgB = RawParseUtils.tagMessage(raw, 0);if 

BLEU: 0.670903540897831


In [ ]:
def batch_translate_examples(examples):
    #Java to CSharp
    inputs = ["translate java to CSharp: " + src for src in examples["java"]]

    #CSharp to Java
    #inputs = ["translate CSharp to java: " + src for src in examples["cs"]]

    tokenized_inputs = tokenizer(
        inputs,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=512
    ).to(device)

    # predictions
    outputs = model.generate(
        **tokenized_inputs,
        max_length=512,
        num_beams=5,
    )

    # outputs to strings
    predictions = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return {"predicted": predictions}

In [ ]:
translated_test_dataset = dataset["test"].map(batch_translate_examples, batched=True, batch_size=8)

Parameter 'function'=<function batch_translate_examples at 0x79e9cf32b2e0> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
preds = translated_test_dataset["predicted"]
refs = translated_test_dataset["java"]

exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

LoRA

In [ ]:
import math
import torch.nn as nn
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from transformers.models.t5.modeling_t5 import T5Attention

class LoRALinear(nn.Linear):
  def __init__(self, in_features, out_features, bias=True, device=None,
               dtype=None, lora_rank=8, lora_alpha=16, lora_dropout=0.1):
    super().__init__(in_features, out_features, bias=bias,
                         device=device, dtype=dtype)
    self.has_weights_merged = False
    if lora_rank > 0:
      self.lora_dropout = nn.Dropout(lora_dropout)
      self.lora_scaling = lora_alpha / lora_rank
      self.lora_A = nn.Parameter(torch.empty(lora_rank, in_features,
                                              device=device, dtype=dtype))
      self.lora_B = nn.Parameter(torch.empty(out_features, lora_rank,
                                              device=device, dtype=dtype))
      self.lora_A.requires_grad = True
      self.lora_B.requires_grad = True
      self.reset_parameters()

  def is_lora(self):
    return hasattr(self, "lora_A")

  def reset_parameters(self):
    nn.Linear.reset_parameters(self)
    if self.is_lora():
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

  def forward(self, input):
    output = super().forward(input)
    if self.is_lora() and not self.has_weights_merged:
      dropped = self.lora_dropout(input)
      A = dropped @ self.lora_A.T
      BA = A @ self.lora_B.T
      output += self.lora_scaling * BA
    return output


  def train(self, mode=True):
    super().train(mode)
    if not self.is_lora():
      return self
    if mode:
      if self.has_weights_merged:
        self.weight.data -= self.lora_scaling * (self.lora_B @ self.lora_A)
        self.has_weights_merged = False
    else:
      if not self.has_weights_merged:
        self.weight.data += self.lora_scaling * (self.lora_B @ self.lora_A)
        self.has_weights_merged = True
    return self


  def eval(self):
    super().eval()
    if self.is_lora() and not self.has_weights_merged:
      self.weight.data += self.lora_scaling * (self.lora_B @ self.lora_A)
      self.has_weights_merged = True
    return self

In [ ]:
def replace_with_lora(model, rank=8, alpha=16, target=("q","v")):
  for module in model.modules():
    if isinstance(module, T5Attention):
      for component in target:
        lin = getattr(module, component)
        lora_lin = LoRALinear(lin.in_features, lin.out_features, bias=False,
                              lora_rank=rank, lora_alpha=alpha,
                              lora_dropout=0.1)
        lora_lin.weight.data = lin.weight.data.clone()
        setattr(module, component, lora_lin)

def freeze_non_lora(model):
  for name, parameter in model.named_parameters():
    if "lora_A" in name or "lora_B" in name:
      parameter.requires_grad = True

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-base")

def encode(datapoint):
  # Java to CSharp
  #model_inputs = tokenizer(datapoint["java"],truncation=True, max_length=512)
  #labels = tokenizer(datapoint["cs"],truncation=True, max_length=512).input_ids

  # CSharp to Java
  model_inputs = tokenizer(datapoint["cs"],truncation=True, max_length=512)
  labels = tokenizer(datapoint["java"],truncation=True, max_length=512).input_ids
  model_inputs["labels"] = labels
  return model_inputs

train_ds = dataset['train'].map(encode, batched=True, remove_columns=dataset['train'].column_names)

tokenizer_config.json:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/703k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/294k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

Map:   0%|          | 0/10300 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-base")
replace_with_lora(model, rank=8, alpha=16)
freeze_non_lora(model)

collator = DataCollatorForSeq2Seq(tokenizer, model=model)
args = TrainingArguments(
    "codet5_lora",
    report_to="none",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=4e-4,
    num_train_epochs=3,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_steps=1000
)

Trainer(model=model,
        args=args,
        train_dataset= train_ds,
        data_collator=collator).train()

model.eval()

model.save_pretrained("codet5_lora_final")
tokenizer.save_pretrained("codet5_lora_final")

config.json:   0%|          | 0.00/1.57k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
100,0.728400
200,0.574700
300,0.534100
400,0.474300
500,0.449300
600,0.396400
700,0.382000
800,0.429600
900,0.366800
1000,0.389200


('codet5_lora_final/tokenizer_config.json',
 'codet5_lora_final/special_tokens_map.json',
 'codet5_lora_final/vocab.json',
 'codet5_lora_final/merges.txt',
 'codet5_lora_final/added_tokens.json',
 'codet5_lora_final/tokenizer.json')

In [ ]:
import evaluate
def javaToCSharp_specificTranslation(tokenizer, model, device,codeString):

  print("Generating Java to C# translation ...")
  source_code = codeString
  prompt = f"Translate Java to C#: {source_code}"
  inputs = tokenizer(prompt, return_tensors="pt", padding="max_length",
                        truncation=True, max_length=512)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  outputs = model.generate(**inputs, max_length=512)
  translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

  return translated_code


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("codet5_lora_final")
tokenizer = AutoTokenizer.from_pretrained("codet5_lora_final")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def translationJavaTest(codeList):
  translationList = []
  for i in range(len(codeList)):
      sourceCodeString = codeList[i]
      translation = javaToCSharp_specificTranslation(tokenizer, model, device, sourceCodeString)
      translationList.append(translation)
  return translationList

javaCodeTestList = runJavaTest(clean=True)
refs = translationJavaTest(javaCodeTestList)
preds = pd.read_csv('javaFinalTest.csv')
preds = preds.iloc[:,0].tolist()
print(preds,refs)
exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

Some weights of the model checkpoint at codet5_lora_final were not used when initializing T5ForConditionalGeneration: ['decoder.block.0.layer.0.SelfAttention.q.lora_A', 'decoder.block.0.layer.0.SelfAttention.q.lora_B', 'decoder.block.0.layer.0.SelfAttention.v.lora_A', 'decoder.block.0.layer.0.SelfAttention.v.lora_B', 'decoder.block.0.layer.1.EncDecAttention.q.lora_A', 'decoder.block.0.layer.1.EncDecAttention.q.lora_B', 'decoder.block.0.layer.1.EncDecAttention.v.lora_A', 'decoder.block.0.layer.1.EncDecAttention.v.lora_B', 'decoder.block.1.layer.0.SelfAttention.q.lora_A', 'decoder.block.1.layer.0.SelfAttention.q.lora_B', 'decoder.block.1.layer.0.SelfAttention.v.lora_A', 'decoder.block.1.layer.0.SelfAttention.v.lora_B', 'decoder.block.1.layer.1.EncDecAttention.q.lora_A', 'decoder.block.1.layer.1.EncDecAttention.q.lora_B', 'decoder.block.1.layer.1.EncDecAttention.v.lora_A', 'decoder.block.1.layer.1.EncDecAttention.v.lora_B', 'decoder.block.10.layer.0.SelfAttention.q.lora_A', 'decoder.block

Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
Generating Java to C# translation ...
['public override void Serialize(ILittleEndianOutput out1){out1.WriteShort(field_1_vcenter);}', 'public virtual void AddAll(NGit.Util.BlockList<T> src){if (src.size == 0){return;}int srcDirIdx = 0;for (; srcDirIdx < src.tailDirIdx; srcDirId

In [ ]:
import evaluate
def csToJava_specificTranslation(tokenizer, model, device,codeString):

  print("Generating C# to Java translation ...")
  source_code = codeString
  prompt = f"Translate C# to Java: {source_code}"
  inputs = tokenizer(prompt, return_tensors="pt", padding="max_length",
                        truncation=True, max_length=512)
  inputs = {k: v.to(device) for k, v in inputs.items()}

  outputs = model.generate(**inputs, max_length=512)
  translated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)

  return translated_code

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("codet5_lora_final")
tokenizer = AutoTokenizer.from_pretrained("codet5_lora_final")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def translationCSTest(codeList):
  translationList = []
  for i in range(len(codeList)):
      sourceCodeString = codeList[i]
      translation = csToJava_specificTranslation(tokenizer, model, device, sourceCodeString)
      translationList.append(translation)
  return translationList

csCodeTestList = runCSTest(clean=True)
refs = translationCSTest(csCodeTestList)
preds = pd.read_csv('csFinalTest.csv')
preds = preds.iloc[:,0].tolist()
print(preds,refs)
exact_matches = sum(p.strip() == r.strip() for p, r in zip(preds, refs))
em_score = exact_matches / len(preds) * 100

print(f"Exact Match: {em_score:.2f}%")

bleu = evaluate.load("bleu")
wrapped_refs = [[ref] for ref in refs]
bleu_score = bleu.compute(predictions=preds, references=wrapped_refs)
print("BLEU:", bleu_score["bleu"])

Some weights of the model checkpoint at codet5_lora_final were not used when initializing T5ForConditionalGeneration: ['decoder.block.0.layer.0.SelfAttention.q.lora_A', 'decoder.block.0.layer.0.SelfAttention.q.lora_B', 'decoder.block.0.layer.0.SelfAttention.v.lora_A', 'decoder.block.0.layer.0.SelfAttention.v.lora_B', 'decoder.block.0.layer.1.EncDecAttention.q.lora_A', 'decoder.block.0.layer.1.EncDecAttention.q.lora_B', 'decoder.block.0.layer.1.EncDecAttention.v.lora_A', 'decoder.block.0.layer.1.EncDecAttention.v.lora_B', 'decoder.block.1.layer.0.SelfAttention.q.lora_A', 'decoder.block.1.layer.0.SelfAttention.q.lora_B', 'decoder.block.1.layer.0.SelfAttention.v.lora_A', 'decoder.block.1.layer.0.SelfAttention.v.lora_B', 'decoder.block.1.layer.1.EncDecAttention.q.lora_A', 'decoder.block.1.layer.1.EncDecAttention.q.lora_B', 'decoder.block.1.layer.1.EncDecAttention.v.lora_A', 'decoder.block.1.layer.1.EncDecAttention.v.lora_B', 'decoder.block.10.layer.0.SelfAttention.q.lora_A', 'decoder.block

Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
Generating C# to Java translation ...
['public void serialize(LittleEndianOutput out) {out.writeShort(field_1_vcenter);}', 'public void addAll(BlockList<T> src) {if (src.size == 0)return;int srcDirIdx = 0;for (; srcDirIdx < src.tailDirIdx; srcDirIdx++)addAll(src.directory[srcDi

BLEU: 0.6817382238665896


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("codet5_lora_final")
tokenizer = AutoTokenizer.from_pretrained("codet5_lora_final")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

#javaToCSharp(tokenizer, model, device)
#cSharpToJava(tokenizer, model, device)

#java code from ocr
java_code = run_specific('java',clean=True)
print(java_code)
translation = javaToCSharp_specificTranslation(tokenizer,model,device,java_code)
print(translation)

Some weights of the model checkpoint at codet5_lora_final were not used when initializing T5ForConditionalGeneration: ['decoder.block.0.layer.0.SelfAttention.q.lora_A', 'decoder.block.0.layer.0.SelfAttention.q.lora_B', 'decoder.block.0.layer.0.SelfAttention.v.lora_A', 'decoder.block.0.layer.0.SelfAttention.v.lora_B', 'decoder.block.0.layer.1.EncDecAttention.q.lora_A', 'decoder.block.0.layer.1.EncDecAttention.q.lora_B', 'decoder.block.0.layer.1.EncDecAttention.v.lora_A', 'decoder.block.0.layer.1.EncDecAttention.v.lora_B', 'decoder.block.1.layer.0.SelfAttention.q.lora_A', 'decoder.block.1.layer.0.SelfAttention.q.lora_B', 'decoder.block.1.layer.0.SelfAttention.v.lora_A', 'decoder.block.1.layer.0.SelfAttention.v.lora_B', 'decoder.block.1.layer.1.EncDecAttention.q.lora_A', 'decoder.block.1.layer.1.EncDecAttention.q.lora_B', 'decoder.block.1.layer.1.EncDecAttention.v.lora_A', 'decoder.block.1.layer.1.EncDecAttention.v.lora_B', 'decoder.block.10.layer.0.SelfAttention.q.lora_A', 'decoder.block

publicintjump(int()nums){intn=nums.length;int[]dp=newint[n];for(inti=1;i<n;i++){intsteps=Integer.MAX_VALUE;for(intj=i−1;j>=0;j--){if(dp[j]!=Integer.MAX_VALUE&&nums[j]+j>=i){steps=Math.min(steps,dp[j]+1);}=dp[i]steps;}}}returndp[n-1];
Generating Java to C# translation ...
public int jump(int()nums) {final int n =nums.length;final int[] dp = new int[n];for (int i = 1; i < n; i++) {intsteps = Integer.MAX_VALUE;for (int j = i−1; j >=0; j--) {if (dp[j] != Integer.MAX_VALUE&&nums[j]+j>=i) {steps = Math.min(steps,dp[j]+1);} = dp[i]steps;}}}return dp[n-1];}

